# Polymer property prediction — Round 2, **v8**

Changes from v7, all of them measured on this data rather than assumed. Diagnostics and the
scripts that produced these numbers are in `diagnostics/`.

**The v7 nested-CV OOF of 0.909 was not honest.** Base columns were fitted on the rows they
were scored on, in three places, and the meta-CV inherited every bit of that:

| leak | where | measured cost |
|---|---|---|
| early stopping on the scoring fold | `train_trees` | +0.012…+0.046 per target, mean **+0.018** |
| best-val-R² checkpoint on the scoring fold | `train_gnn_fold`, `train_multitask` | **+0.04…+0.07** on the ~220-row targets |
| feature selection fitted on 100% of labels | `select_features` | +0.008 (eea), +0.016 (ei) |

That is the entire 0.909 → 0.887 OOF/LB gap. v8 fixes all three, so **expect the reported OOF to
FALL to roughly 0.89 — that is the fix working, not a regression.** It should now track the
leaderboard within ~0.006, the metric's own sampling sd at these test sizes.

**What was ruled out** (don't re-investigate): cross-target sibling availability is 96–97% on
both train and test; nearest-neighbour Tanimoto is 0.55 train-internal vs 0.56 test→train, so the
folds are a faithful simulation; there are 3 conflicting label groups in 7406 and 0 rows dropped;
and no train/test molecule shares a target.

**Genuine additions:**
- **Tanimoto/MinMax-kernel GP** on every target — blend gain +0.026 (eea), +0.023 (egb),
  +0.010 (ei). Keep it even where its own OOF is below the trees; the value is decorrelation.
- **Polymer blocks** (trimer, backbone/side-chain, conjugation length, SMARTS) gated to
  eea/egb/egc — +0.012 and +0.017 there, but −0.012 on nc and −0.009 on eps, hence the gate.
- **Shrunk stack weights** — inner CV picks pure equal weighting on eea/ei/egb; NNLS weights are
  noise at n≈220.
- **10 seeds instead of 3** on the small targets.

**Tried and rejected, with numbers:** a wide feature dump (~1500 cols) cost nc and eps 0.033 each;
AUTOCORR2D scored below baseline on 4 of 5 targets; PLS hit 0.56 on eps and dragged blends
negative; Lorentz–Lorenz has univariate R²=0.59 on nc but *negative* incremental value because the
trees already reconstruct it.

Realistic honest OOF for this pipeline is **~0.90**. ei (≈0.82) and eps (≈0.78) did not move for
any feature block, model class or blend tested, and they cap the equal-weight mean.

## 1. Setup & Imports

In [1]:
import importlib, subprocess, sys



def ensure(import_name, pip_name=None):

    try:

        importlib.import_module(import_name)

    except ImportError:

        print(f"Installing {pip_name or import_name} ...")

        subprocess.run([sys.executable, "-m", "pip", "install", "-q", pip_name or import_name], check=True)



for imp, pip in [("rdkit","rdkit"), ("xgboost","xgboost"),

                 ("lightgbm","lightgbm"), ("catboost","catboost"), ("torch","torch")]:

    ensure(imp, pip)



import os, warnings, numpy as np, pandas as pd

from sklearn.metrics import r2_score, mean_absolute_error

from sklearn.linear_model import Ridge

from sklearn.model_selection import KFold

from scipy.optimize import nnls

import xgboost as xgb, lightgbm as lgb

from catboost import CatBoostRegressor

import torch, torch.nn as nn, torch.nn.functional as F



from rdkit import Chem

from rdkit.Chem import Descriptors, AllChem, MACCSkeys, rdMolDescriptors

from rdkit.Avalon import pyAvalonTools

from rdkit import DataStructs, RDLogger

RDLogger.DisableLog("rdApp.*")

warnings.filterwarnings("ignore")



SEED = 42

np.random.seed(SEED); torch.manual_seed(SEED)

DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")

print("Imports OK. Torch device:", DEVICE)


Installing rdkit ...
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 37.4/37.4 MB 57.2 MB/s eta 0:00:00
Imports OK. Torch device: cuda


## 2. Load Data & Normalize Schema



Handles the usual layout `id, smiles, target_type, target` and also a wide `Tg`/`Egc`

two-column format (melted into long form).

In [2]:
DATA_DIR = "/kaggle/input/competitions/ppp-round-2"   # <-- set to the real Kaggle path

TRAIN_PATH = os.path.join(DATA_DIR, "train.csv")

TEST_PATH  = os.path.join(DATA_DIR, "test.csv")

if not os.path.exists(TRAIN_PATH):

    TRAIN_PATH, TEST_PATH = "train.csv", "test.csv"

if not os.path.exists(TRAIN_PATH):

    TRAIN_PATH, TEST_PATH = "/mnt/user-data/uploads/train.csv", "/mnt/user-data/uploads/test.csv"

# PI1M auxiliary corpus (used for self-supervised GNN pretraining; see 5c)

PI1M_PATH = os.path.join(DATA_DIR, "PI1M.csv")

for _cand in ("PI1M.csv", "/mnt/user-data/uploads/PI1M.csv"):

    if not os.path.exists(PI1M_PATH) and os.path.exists(_cand): PI1M_PATH = _cand



train = pd.read_csv(TRAIN_PATH)

test  = pd.read_csv(TEST_PATH)

print("Train:", train.shape, "| Test:", test.shape)



def normalize_schema(df, is_train):

    df = df.copy(); low = {c.lower(): c for c in df.columns}

    idc = next((low[c] for c in ["id","index"] if c in low), None)

    if idc is None: df.insert(0,"id",np.arange(len(df)))

    elif idc!="id": df = df.rename(columns={idc:"id"})

    sc = next((low[c] for c in ["smiles","smile","canonical_smiles"] if c in low), None)

    if sc is None: raise ValueError(f"No SMILES column in {list(df.columns)}")

    if sc!="smiles": df = df.rename(columns={sc:"smiles"})

    ttc = next((low[c] for c in ["target_type","property","property_type","task"] if c in low), None)

    if ttc and ttc!="target_type": df = df.rename(columns={ttc:"target_type"})

    if "target_type" in df.columns:

        # PHASE 2: 7 properties (tg, egc, egb, eps, nc, ei, eea). Keep the label verbatim

        # (lower-cased) rather than forcing the old {Tg,Egc} map -- otherwise 5 of 7 targets

        # silently vanish and the submission is blank for ~1k test rows.

        df["target_type"] = df["target_type"].astype(str).str.strip().str.lower()

    elif is_train:

        prop_cols = [c for c in df.columns if c.lower() not in ("id","smiles","smiles_canon")]

        rows=[]

        for _,r in df.iterrows():

            base={"id":r.get("id"),"smiles":r["smiles"]}

            for pc in prop_cols:

                if pd.notna(r[pc]): rows.append({**base,"target_type":pc.lower(),"target":r[pc]})

        df = pd.DataFrame(rows).reset_index(drop=True)

        print("Melted wide -> long over", prop_cols)

    if is_train and "target" not in df.columns:

        low = {c.lower(): c for c in df.columns}

        tc = next((low[c] for c in ["target","value","y"] if c in low), None)

        if tc is None: raise ValueError(f"No target column in {list(df.columns)}")

        if tc!="target": df = df.rename(columns={tc:"target"})

    return df



train = normalize_schema(train, True)

test  = normalize_schema(test, False)

print("Train cols:", list(train.columns), "| Test cols:", list(test.columns))

TARGETS = sorted(train["target_type"].unique().tolist())

print("Discovered TARGETS:", TARGETS)

print("Train counts:", train["target_type"].value_counts().to_dict())

print("Test  counts:", test["target_type"].value_counts().to_dict())



# per-target size tiers: small targets (~220 rows) get a lean, regularized pipeline to avoid overfit

_counts = train["target_type"].value_counts().to_dict()

SMALL_THRESHOLD = 600

BIG_TARGETS   = [t for t in TARGETS if _counts.get(t,0) >= SMALL_THRESHOLD]

SMALL_TARGETS = [t for t in TARGETS if _counts.get(t,0) <  SMALL_THRESHOLD]

print(f"BIG (full stack): {BIG_TARGETS}")

print(f"SMALL (lean stack): {SMALL_TARGETS}")



# metric note: competition scores per-target R2 then averages. tg spans ~-110..495 while nc spans

# ~1.5..2.8, so every model below is trained and scored strictly per target (never pooled).


Train: (7409, 3) | Test: (4940, 3)
Train cols: ['id', 'smiles', 'target', 'target_type'] | Test cols: ['id', 'smiles', 'target_type']
Discovered TARGETS: ['eea', 'egb', 'egc', 'ei', 'eps', 'nc', 'tg']
Train counts: {'tg': 4143, 'egc': 2028, 'egb': 337, 'eps': 229, 'nc': 229, 'ei': 222, 'eea': 221}
Test  counts: {'tg': 2763, 'egc': 1352, 'egb': 224, 'eps': 153, 'nc': 153, 'ei': 148, 'eea': 147}
BIG (full stack): ['egc', 'tg']
SMALL (lean stack): ['eea', 'egb', 'ei', 'eps', 'nc']


## 2b. Canonicalize, Resolve Conflicts, Build Leak-Safe Group Folds



Every appearance of the same molecule (canonical SMILES) is forced into the same fold, so

CV never trains on a molecule it's about to score. Conflicting duplicate labels collapse to

the per-group median. **Change from v1: 8 folds instead of 5** — with only ~2-4k rows per

target, 5-fold OOF estimates are noisier than they look; 8 folds gives a steadier signal to

tune against and to fit stacking weights on.

In [3]:
def canonical(smi):

    m = Chem.MolFromSmiles(str(smi))

    return Chem.MolToSmiles(m) if m is not None else None



for df in (train, test):

    df["smiles_canon"] = df["smiles"].apply(canonical)

    df["smiles_canon"] = df["smiles_canon"].fillna(df["smiles"])



bad = train["smiles"].apply(lambda s: Chem.MolFromSmiles(str(s)) is None)

if bad.any():

    print(f"Dropping {int(bad.sum())} unparseable train SMILES.")

    train = train[~bad].reset_index(drop=True)



key = ["smiles_canon","target_type"]

conf = (train.groupby(key)["target"].nunique() > 1).sum()

print(f"Conflicting duplicate groups: {conf}")



# NEW: for badly conflicting duplicates (large spread), drop instead of blindly medianing —

# a bad label quietly caps achievable R2.

grp = train.groupby(key)["target"]

spread = grp.transform(lambda s: s.max()-s.min())

scale = train.groupby("target_type")["target"].transform(lambda s: s.std())

bad_conflict = (spread > 0) & (spread > 0.5*scale)

print(f"Dropping {int(bad_conflict.sum())} rows from high-spread conflicting duplicate groups.")

train = train[~bad_conflict].reset_index(drop=True)



train["target"] = train.groupby(key)["target"].transform("median")

before = len(train)

train = train.drop_duplicates(subset=key, keep="first").reset_index(drop=True)

print(f"Deduped {before} -> {len(train)} rows.")



train["row_id"] = np.arange(len(train))

test["row_id"]  = np.arange(len(test))



N_FOLDS = 8

def make_group_folds(groups, n_splits, seed=SEED):

    groups = np.asarray(groups)

    uniq, sizes = np.unique(groups, return_counts=True)

    rng = np.random.RandomState(seed); perm = rng.permutation(len(uniq))

    uniq, sizes = uniq[perm], sizes[perm]

    order = np.argsort(-sizes); load = np.zeros(n_splits, dtype=int); g2f={}

    for gi in order:

        f = int(np.argmin(load)); g2f[uniq[gi]] = f; load[f]+=sizes[gi]

    fid = np.array([g2f[g] for g in groups]); idx=np.arange(len(groups))

    return [(idx[fid!=f], idx[fid==f]) for f in range(n_splits)]



print("Per-target rows:", train["target_type"].value_counts().to_dict())


Conflicting duplicate groups: 4
Dropping 0 rows from high-spread conflicting duplicate groups.
Deduped 7409 -> 7405 rows.
Per-target rows: {'tg': 4139, 'egc': 2028, 'egb': 337, 'eps': 229, 'nc': 229, 'ei': 222, 'eea': 221}


## 3. Featurization (fingerprints + descriptors + polymer group-contribution features)



Complementary structural views: **Morgan counts at r=2 and r=3**, **Avalon** and **MACCS**,

the full **RDKit 2D descriptor** block, and an **expanded polymer-backbone / group-contribution**

block. Tg/Egc are well-studied physically (Bicerano-style group contribution), so features like

H-bond density, ring fraction, rotatable-bond density, and molecular weight per repeat unit

carry direct physical signal beyond what generic fingerprints capture.

In [4]:
DESC_FUNCS = [(n,f) for n,f in Descriptors.descList]

print("RDKit 2D descriptors:", len(DESC_FUNCS))



def desc_2d(m):

    out=[]

    for _,f in DESC_FUNCS:

        try: out.append(f(m))

        except Exception: out.append(np.nan)

    return out



def morgan_counts(m, radius, nbits):

    fp = AllChem.GetHashedMorganFingerprint(m, radius=radius, nBits=nbits)

    a = np.zeros(nbits, dtype=np.int16)

    for i,c in fp.GetNonzeroElements().items(): a[i]=c

    return a



def avalon_fp(m, nbits=1024):

    a = np.zeros(nbits, dtype=np.int8)

    DataStructs.ConvertToNumpyArray(pyAvalonTools.GetAvalonFP(m, nBits=nbits), a)

    return a



def maccs_fp(m):

    a = np.zeros(167, dtype=np.int8)

    DataStructs.ConvertToNumpyArray(MACCSkeys.GenMACCSKeys(m), a)

    return a



def backbone_feats(m, smi):

    stars = [a.GetIdx() for a in m.GetAtoms() if a.GetSymbol()=="*"]

    path_len, conj_frac = np.nan, np.nan

    if len(stars) >= 2:

        try:

            p = Chem.GetShortestPath(m, stars[0], stars[1])

            if p and len(p) >= 2:

                path_len = len(p)-1

                nc = sum(1 for i in range(len(p)-1)

                         if (b:=m.GetBondBetweenAtoms(p[i],p[i+1])) is not None and b.GetIsConjugated())

                conj_frac = nc/path_len

        except Exception: pass

    na = m.GetNumAtoms()

    n_ar = sum(a.GetIsAromatic() for a in m.GetAtoms())

    n_hbd = rdMolDescriptors.CalcNumHBD(m)

    n_hba = rdMolDescriptors.CalcNumHBA(m)

    mw = Descriptors.MolWt(m)

    n_ring = rdMolDescriptors.CalcNumRings(m)

    n_arom_ring = rdMolDescriptors.CalcNumAromaticRings(m)

    n_rot = rdMolDescriptors.CalcNumRotatableBonds(m)

    fsp3 = rdMolDescriptors.CalcFractionCSP3(m)

    n_heavy = m.GetNumHeavyAtoms()

    tpsa = Descriptors.TPSA(m)

    return [n_ar, sum(a.GetSymbol()=="O" for a in m.GetAtoms()),

            sum(a.GetSymbol()=="N" for a in m.GetAtoms()),

            sum(a.GetSymbol()=="S" for a in m.GetAtoms()),

            str(smi).count("*"), n_ar/na if na else 0.0,

            n_rot, n_ring, fsp3, path_len, conj_frac,

            n_hbd, n_hba, mw, n_arom_ring,

            (n_hbd+n_hba)/n_heavy if n_heavy else 0.0,   # H-bond density (Bicerano-style cohesion proxy)

            n_rot/n_heavy if n_heavy else 0.0,           # rotatable-bond density -> chain flexibility

            n_ring/n_heavy if n_heavy else 0.0,          # ring density -> chain stiffness proxy

            tpsa, tpsa/n_heavy if n_heavy else 0.0,

            mw/(path_len+1) if not np.isnan(path_len) else np.nan]  # rough MW-per-backbone-length

EXTRA = ["n_arom","n_O","n_N","n_S","n_star","arom_frac","n_rotbond",

         "n_rings","fsp3","bb_path_len","bb_conj_frac","n_hbd","n_hba","mw",

         "n_arom_ring","hbond_density","rotbond_density","ring_density",

         "tpsa","tpsa_density","mw_per_backbone"]



MORGAN_BITS = 1024   # reduced from 2048 -> less noise dilution given row count

def featurize(df):

    D,M2,M3,AV,MA,EX = [],[],[],[],[],[]

    for s in df["smiles"]:

        m = Chem.MolFromSmiles(str(s))

        if m is None:

            D.append([np.nan]*len(DESC_FUNCS)); M2.append(np.zeros(MORGAN_BITS,np.int16))

            M3.append(np.zeros(MORGAN_BITS,np.int16)); AV.append(np.zeros(1024,np.int8))

            MA.append(np.zeros(167,np.int8)); EX.append([np.nan]*len(EXTRA)); continue

        D.append(desc_2d(m)); M2.append(morgan_counts(m,2,MORGAN_BITS))

        M3.append(morgan_counts(m,3,MORGAN_BITS)); AV.append(avalon_fp(m)); MA.append(maccs_fp(m))

        EX.append(backbone_feats(m,s))

    dcol=[f"d_{n}" for n,_ in DESC_FUNCS]; excol=[f"x_{c}" for c in EXTRA]

    m2col=[f"m2_{i}" for i in range(MORGAN_BITS)]; m3col=[f"m3_{i}" for i in range(MORGAN_BITS)]

    avcol=[f"av_{i}" for i in range(1024)]; macol=[f"ma_{i}" for i in range(167)]

    X = np.hstack([np.array(D,dtype=np.float32), np.array(EX,dtype=np.float32),

                   np.array(M2), np.array(M3), np.array(AV), np.array(MA)])

    cols = dcol+excol+m2col+m3col+avcol+macol

    return pd.DataFrame(X, columns=cols), dcol+excol, m2col+m3col+avcol+macol



print("Featurizing train...")

Xtr_df, DENSE_COLS, SPARSE_COLS = featurize(train)

print("Featurizing test...")

Xte_df, _, _ = featurize(test)

print(Xtr_df.shape, Xte_df.shape)


RDKit 2D descriptors: 217
Featurizing train...
Featurizing test...
(7405, 3477) (4940, 3477)


### 3b. Clean & prune features

In [5]:
ALL_COLS = DENSE_COLS + SPARSE_COLS

CLIP = 1e6

def sanitize(df, cols, med=None):

    df = df.copy()

    df[cols] = df[cols].replace([np.inf,-np.inf], np.nan)

    if med is None: med = df[cols].median().fillna(0.0)

    df[cols] = df[cols].fillna(med).fillna(0.0).clip(-CLIP, CLIP)

    return df, med



Xtr_df, med = sanitize(Xtr_df, DENSE_COLS)

Xte_df, _   = sanitize(Xte_df, DENSE_COLS, med)



dv = Xtr_df[DENSE_COLS].var(); DENSE_COLS=[c for c in DENSE_COLS if dv[c]>0]

sv = Xtr_df[SPARSE_COLS].var(); SPARSE_COLS=[c for c in SPARSE_COLS if sv[c]>1e-4]

ALL_COLS = DENSE_COLS + SPARSE_COLS

for d in (Xtr_df, Xte_df):

    assert not np.isinf(d[ALL_COLS].values).any() and not np.isnan(d[ALL_COLS].values).any()

print(f"After variance pruning: {len(ALL_COLS)} features ({len(DENSE_COLS)} dense + {len(SPARSE_COLS)} sparse)")



Xtr_all = Xtr_df[ALL_COLS].values.astype(np.float32)

Xte_all = Xte_df[ALL_COLS].values.astype(np.float32)


After variance pruning: 3445 features (217 dense + 3228 sparse)


### 3i. Cross-target features (N-way, leak-safe) — the multi-property lever



415 training molecules carry 2-6 of the 7 properties, so a molecule's *other* measured properties

predict the one being modelled (Tg and the band-gap family are physically coupled). For each row we add

every *other* property's value when known for that molecule (leak-safe out-of-fold lookup on train;

direct lookup else all-train LGBM cross-predictor on test), plus a `*_known` flag. The row's own target

is zeroed so it can never leak. This is the single highest-value tabular block for phase 2 and what makes

the small targets learnable. It runs *before* feature selection so the selector can keep these columns.

In [6]:
# ===== N-way cross-target features (leak-safe) =====

# FIX vs v5: the v5 train-side lookup was built per modeling-fold, so a molecule's OWN sibling

# properties landed in the same fold as the row consuming them and got excluded -> train `known`

# was ~0 and OOF was BLIND to the single most predictive signal for eea/ei/eps/nc (data analysis

# showed 98-99% of their test molecules have 2-3 correlated siblings known in train).

# KEY INSIGHT: looking up a DIFFERENT property of the same molecule is NOT label leakage -- the

# egc value is not the eps label. The only thing that would leak is the row's OWN (molecule,target),

# which the self-column zeroing already removes. So sibling lookup needs no out-of-fold split at all;

# it can use the full lookup table on both train and test. This makes CV truthful about the small targets.

_canon_tr = train["smiles_canon"].values; _tt_tr = train["target_type"].values

_y_tr = train["target"].values.astype(np.float32)

_canon_te = test["smiles_canon"].values;  _tt_te = test["target_type"].values



_LUT = {t: {} for t in TARGETS}

for c,t,v in zip(_canon_tr,_tt_tr,_y_tr): _LUT[t][c]=v

_MED = {t:(float(np.median(list(_LUT[t].values()))) if _LUT[t] else 0.0) for t in TARGETS}



def _cross_block(canon):

    """Full-table sibling lookup for every row. Self-column is zeroed afterwards, so no row ever

       reads its own target -> no leakage. Same code path for train and test."""

    n=len(canon); T=len(TARGETS)

    vals=np.zeros((n,T),np.float32); known=np.zeros((n,T),np.float32)

    for j,t in enumerate(TARGETS):

        lut=_LUT[t]

        for i2,c in enumerate(canon):

            if c in lut: vals[i2,j]=lut[c]; known[i2,j]=1.0

            else:        vals[i2,j]=_MED[t]

    return vals,known



print("Building cross-target lookup features (full-table sibling lookup) ...")

CROSS_tr_val,CROSS_tr_known = _cross_block(_canon_tr)

CROSS_te_val,CROSS_te_known = _cross_block(_canon_te)



# fill still-unknown cells with cross-predictors. For TRAIN these are fit out-of-fold (a predicted

# sibling value derived from features could otherwise peek), for TEST fit on all train.

print("Filling unknown cross cells with cross-predictors ...")

def _mk_cross(): return lgb.LGBMRegressor(n_estimators=500,learning_rate=0.03,num_leaves=63,

    subsample=0.85,colsample_bytree=0.4,reg_lambda=1.0,random_state=SEED,n_jobs=-1,verbosity=-1)

for fi_,(tr,va) in enumerate(make_group_folds(_canon_tr,N_FOLDS,seed=SEED)):

    for j,t in enumerate(TARGETS):

        need=va[(CROSS_tr_known[va,j]==0.0)]

        if len(need)==0: continue

        src=tr[_tt_tr[tr]==t]

        if len(src)<30: continue

        m=_mk_cross().fit(Xtr_all[src],_y_tr[src]); CROSS_tr_val[need,j]=m.predict(Xtr_all[need])

_full_cross={}

for t in TARGETS:

    src=np.where(_tt_tr==t)[0]

    if len(src)>=30: _full_cross[t]=_mk_cross().fit(Xtr_all[src],_y_tr[src])

for j,t in enumerate(TARGETS):

    if t not in _full_cross: continue

    need=np.where(CROSS_te_known[:,j]==0.0)[0]

    if len(need): CROSS_te_val[need,j]=_full_cross[t].predict(Xte_all[need])



# zero the row's OWN target column: the ONLY genuine leak path

_self={t:j for j,t in enumerate(TARGETS)}

for i2 in range(len(_tt_tr)):

    j=_self[_tt_tr[i2]]; CROSS_tr_val[i2,j]=_MED[TARGETS[j]]; CROSS_tr_known[i2,j]=0.0

for i2 in range(len(_tt_te)):

    j=_self[_tt_te[i2]]; CROSS_te_val[i2,j]=_MED[TARGETS[j]]; CROSS_te_known[i2,j]=0.0



Xtr_all=np.hstack([Xtr_all,CROSS_tr_val,CROSS_tr_known]).astype(np.float32)

Xte_all=np.hstack([Xte_all,CROSS_te_val,CROSS_te_known]).astype(np.float32)

ALL_COLS=ALL_COLS+[f"cross_{t}" for t in TARGETS]+[f"cross_{t}_known" for t in TARGETS]

DENSE_COLS=DENSE_COLS+[f"cross_{t}" for t in TARGETS]+[f"cross_{t}_known" for t in TARGETS]

print(f"Added {2*len(TARGETS)} cross-target features -> total {Xtr_all.shape[1]}. "

      f"Known cells: train={int(CROSS_tr_known.sum())}, test={int(CROSS_te_known.sum())}")


Building cross-target lookup features (full-table sibling lookup) ...
Filling unknown cross cells with cross-predictors ...
Added 14 cross-target features -> total 3459. Known cells: train=4954, test=3484


### 3j. Polymer blocks (trimer / backbone / conjugation), gated to electronic targets

In [ ]:
# ===== 3j. Polymer-specific blocks -- gated to the electronic targets (NEW in v8) =====
# Built from the two '*' attachment points: a head-to-tail TRIMER (so Morgan environments at the
# repeat boundary are real chemistry instead of a dummy atom, and conjugation across the junction
# becomes visible), a backbone/side-chain split along the shortest path between the stars, a
# conjugation-length block, and donor/acceptor + group-contribution SMARTS counts.
#
# MEASURED, and the gating matters a lot:
#     eea  +0.0119   egb  +0.0171        <- electronic targets, real gain
#     ei   +0.0009                       <- neutral
#     nc   -0.0117   eps  -0.0087        <- HURTS, so they are excluded
# Also measured: delivering the same signal as a wide block (adding 192 AUTOCORR2D columns) scored
# BELOW baseline on 4 of 5 targets. Width itself is the problem below ~350 rows, so this block is
# deliberately kept to 77 columns and applied to 3 of 7 targets only.
from rdkit.Chem import Crippen

POLY_TARGETS = {"eea", "egb", "egc"}          # egc is the same physics family as egb

_VDW={'C':20.58,'N':15.60,'O':14.71,'S':24.43,'F':13.31,'Cl':22.45,'Br':26.52,'I':32.52,
      'Si':38.79,'P':24.43,'B':21.0,'*':0.0,'H':7.24}
def _vol(m): return sum(_VDW.get(a.GetSymbol(),18.0) for a in Chem.AddHs(m).GetAtoms())
def _starsof(m): return [a.GetIdx() for a in m.GetAtoms() if a.GetAtomicNum()==0]

def build_oligomer(smi, n=3):
    """Head-to-tail n-mer from a 2-star repeat unit, ends capped with H. Parses 6565/6565
       of the training molecules."""
    m=Chem.MolFromSmiles(str(smi))
    if m is None or len(_starsof(m))!=2: return None
    combo=m
    for _ in range(n-1): combo=Chem.CombineMols(combo,m)
    rw=Chem.RWMol(combo); na=m.GetNumAtoms(); sl=_starsof(m)
    cps=[[s+k*na for s in sl] for k in range(n)]
    def nbr(s):
        x=[a.GetIdx() for a in rw.GetAtomWithIdx(s).GetNeighbors()]
        return x[0] if len(x)==1 else None
    for k in range(n-1):
        a,b=nbr(cps[k][1]),nbr(cps[k+1][0])
        if a is None or b is None: return None
        if rw.GetBondBetweenAtoms(a,b) is None: rw.AddBond(a,b,Chem.BondType.SINGLE)
    keep={cps[0][0],cps[n-1][1]}
    for s in sorted([x for c in cps for x in c if x not in keep],reverse=True): rw.RemoveAtom(s)
    try:
        o=rw.GetMol(); Chem.SanitizeMol(o)
        return Chem.MolFromSmiles(Chem.MolToSmiles(o).replace("*","[H]"))
    except Exception: return None

def _conj_max(m):
    """Size of the largest connected conjugated system -- a particle-in-a-box gap proxy."""
    if m is None: return 0
    par={}
    def find(x):
        while par.get(x,x)!=x: x=par.get(x,x)
        return x
    ats=set()
    for b in m.GetBonds():
        if b.GetIsConjugated():
            i,j=b.GetBeginAtomIdx(),b.GetEndAtomIdx(); ats.add(i); ats.add(j)
            par.setdefault(i,i); par.setdefault(j,j)
            ri,rj=find(i),find(j)
            if ri!=rj: par[ri]=rj
    if not ats: return 0
    from collections import Counter
    return max(Counter(find(a) for a in ats).values())

_DONOR=["[NX3;H2,H1,H0;!$(N[C,S]=[O,S,N])]","[OX2;!$(O=*);!$(O[C]=O)][CX4]","c1ccsc1","[SX2][CX4]",
        "[cH0]([CX4])","[OX2]c","[NX3]c","c1cc[nH]c1","[Se]","[SiX4]"]
_ACCEPT=["[NX3](=O)=O","[CX2]#[NX1]","[CX3]=[OX1]","[CX3](=O)[OX2]","[CX3](=O)[NX3]","[SX4](=O)(=O)",
         "[F][CX4]","c1nsnc1","[CX3]=[NX2]","[n+]","[CX4]([F])([F])[F]","[PX4]=O"]
_GROUPS=["[CX3](=O)[OX2H0][#6]","[CX3](=O)[NX3]","[NX3][CX3](=O)[OX2]","[#6][OX2][#6]","[OX2][CX3](=O)[OX2]",
         "[SX4](=O)(=O)[#6]","O=C1[NX3][CX3](=O)c2ccccc12","[SiX4][OX2]","c1ccccc1","c1ccc2ccccc2c1",
         "[CX4]([CH3])([CH3])","[CX4]([F])([F])[F]","[CX2]#[NX1]","[NX3H2]","[OX2H]","[CX4H2][CX4H2][CX4H2]"]
_SMA=[s for s in (Chem.MolFromSmarts(x) for x in _DONOR+_ACCEPT+_GROUPS) if s is not None]

def poly_block(smi):
    m=Chem.MolFromSmiles(str(smi))
    if m is None: return None
    hv=m.GetNumHeavyAtoms() or 1
    mr=Crippen.MolMR(m); v=_vol(m); ll=mr/v if v>0 else 0.0
    npred=np.sqrt((1+2*ll)/(1-ll)) if 0<ll<1 else 0.0
    PHYS=[mr,v,ll,npred,mr/hv,Descriptors.TPSA(m)/v if v else 0.0,
          Crippen.MolLogP(m)/hv,Descriptors.MolWt(m)/v if v else 0.0]
    st=_starsof(m); bbl=0; bb=set()
    if len(st)==2:
        try:
            p=Chem.GetShortestPath(m,st[0],st[1]); bb=set(p)-set(st); bbl=max(len(p)-2,0)
        except Exception: pass
    sc=[a.GetIdx() for a in m.GetAtoms() if a.GetIdx() not in bb and a.GetAtomicNum()>0]
    def part(ix):
        if not ix: return [0.0]*6
        As=[m.GetAtomWithIdx(i) for i in ix]
        return [len(ix), sum(a.GetIsAromatic() for a in As)/len(ix),
                sum(a.IsInRing() for a in As)/len(ix),
                sum(a.GetAtomicNum() not in (6,1) for a in As)/len(ix),
                sum(a.GetMass() for a in As),
                sum(a.GetHybridization()==Chem.rdchem.HybridizationType.SP3 for a in As)/len(ix)]
    bbf,scf=part(sorted(bb)),part(sc)
    nrb=sum(1 for b in m.GetBonds() if b.GetBeginAtomIdx() in bb and b.GetEndAtomIdx() in bb
            and not b.IsInRing() and b.GetBondType()==Chem.BondType.SINGLE)
    STRUCT=bbf+scf+[bbl,nrb,nrb/max(bbl,1),len(sc)/hv,bbf[4]/max(scf[4],1e-6)]
    cm=_conj_max(m); tri=build_oligomer(smi,3); ct=_conj_max(tri) if tri is not None else cm
    CONJ=[cm,cm/hv,1.0/max(cm,1),ct,ct/max(cm,1),1.0/max(ct,1)]
    if tri is not None:
        thv=tri.GetNumHeavyAtoms() or 1; tmr=Crippen.MolMR(tri); tv=_vol(tri)
        tll=tmr/tv if tv>0 else 0.0
        TRI=[thv,Descriptors.TPSA(tri)/thv,rdMolDescriptors.CalcFractionCSP3(tri),
             rdMolDescriptors.CalcNumRotatableBonds(tri)/thv,tll,
             np.sqrt((1+2*tll)/(1-tll)) if 0<tll<1 else 0.0,
             sum(a.GetIsAromatic() for a in tri.GetAtoms())/thv,
             rdMolDescriptors.CalcNumAromaticRings(tri)]
    else: TRI=[0.0]*8
    SM=[len(m.GetSubstructMatches(s))/hv for s in _SMA]
    return PHYS+STRUCT+CONJ+TRI+SM

print("Building polymer blocks (trimer / backbone / conjugation / SMARTS) ...")
_cache={}
def _poly_rows(df):
    rows=[]
    for s in df["smiles"]:
        if s not in _cache: _cache[s]=poly_block(s)
        rows.append(_cache[s])
    width=max((len(r) for r in rows if r), default=1)
    rows=[r if r else [0.0]*width for r in rows]
    A=np.array(rows,dtype=np.float32)
    return np.nan_to_num(A,nan=0.0,posinf=0.0,neginf=0.0).clip(-1e6,1e6)

POLY_tr=_poly_rows(train); POLY_te=_poly_rows(test)
print(f"Polymer block: {POLY_tr.shape[1]} columns, applied to {sorted(POLY_TARGETS)} only.")

def get_X(target):
    """Per-target design matrix. The polymer block is appended only for the electronic targets."""
    if target in POLY_TARGETS:
        return (np.hstack([Xtr_all, POLY_tr]).astype(np.float32),
                np.hstack([Xte_all, POLY_te]).astype(np.float32))
    return Xtr_all, Xte_all

### 3c. Per-target feature selection (NEW)



With ~5,500 raw features and only a few thousand rows per target, most of that width is noise

that dilutes tree splits and inflates variance. Fit a quick LightGBM pass per target, rank

features by gain, and keep only the informative subset — separately for Tg and Egc since they

likely respond to different structural cues.

In [ ]:
# ===== 3c. Per-target feature selection -- v8: IN-FOLD for big targets, NONE for small =====
# Measured on this data (8-fold, honest protocol). Fitting the selector on 100% of a target's
# labels and reusing that subset in every fold inflated OOF by +0.008 (eea) and +0.016 (ei)
# with no test-time benefit. On eps and nc, honest in-fold selection scored HIGHER than the
# leaky global one -- i.e. below ~600 rows the selection was not buying accuracy at all, only
# optimism. So: big targets select inside the fold, small targets keep full width and lean on
# colsample_bytree + regularisation.

TOP_K = 900

def select_features_infold(X, y, target, seed=SEED):
    """Rank features by LightGBM gain using ONLY the rows handed in (always a training fold)."""
    if target in SMALL_TARGETS:
        return np.arange(X.shape[1])                      # no selection below SMALL_THRESHOLD
    probe = lgb.LGBMRegressor(n_estimators=400, learning_rate=0.05, num_leaves=63,
                              subsample=0.8, colsample_bytree=0.6,
                              random_state=seed, verbosity=-1, n_jobs=-1)
    probe.fit(X, y)
    return np.sort(np.argsort(-probe.feature_importances_)[:min(TOP_K, X.shape[1])])

print("Feature selection is now in-fold (big targets) / disabled (small targets).")
print("  BIG  :", BIG_TARGETS, f"-> top {TOP_K} by in-fold gain")
print("  SMALL:", SMALL_TARGETS, "-> full width, no selection")

## 4. Tree Ensemble (XGBoost + LightGBM + CatBoost), seed-bagged



Grouped 8-fold CV repeated over 3 seeds and averaged, now on the per-target selected feature

subset. Produces OOF + test predictions per base model for later stacking.

In [ ]:
from sklearn.preprocessing import PowerTransformer
class _RawT:
    def fit_transform(self,y): return y.ravel()
    def transform(self,y): return y.ravel()
    def inverse_transform(self,y): return y.ravel()
def _make_t(mode): return PowerTransformer(method="yeo-johnson") if mode=="yj" else _RawT()

def get_models(seed):
    return {
        "xgb": xgb.XGBRegressor(n_estimators=3000, learning_rate=0.02, max_depth=6,
                subsample=0.8, colsample_bytree=0.3, colsample_bylevel=0.5, reg_alpha=0.2,
                reg_lambda=1.5, min_child_weight=3, random_state=seed, n_jobs=-1, tree_method="hist",
                early_stopping_rounds=120, eval_metric="rmse"),
        "lgb": lgb.LGBMRegressor(n_estimators=3000, learning_rate=0.02, num_leaves=63,
                subsample=0.8, subsample_freq=1, colsample_bytree=0.3, reg_alpha=0.2,
                reg_lambda=1.5, min_child_samples=20, random_state=seed, n_jobs=-1, verbosity=-1),
        "lgb_et": lgb.LGBMRegressor(n_estimators=3000, learning_rate=0.02, num_leaves=127,
                subsample=0.7, subsample_freq=1, colsample_bytree=0.25, reg_alpha=0.1, reg_lambda=1.0,
                min_child_samples=10, extra_trees=True, random_state=seed+7, n_jobs=-1, verbosity=-1),
        "cat": CatBoostRegressor(iterations=3000, learning_rate=0.02, depth=6, l2_leaf_reg=3.0,
                rsm=0.3, random_seed=seed, verbose=False, early_stopping_rounds=120),
    }
BIG_TREE_NAMES = ["xgb","lgb","lgb_et","cat"]
BIG_TRANSFORM  = {"xgb":"yj","lgb":"raw","lgb_et":"yj","cat":"yj"}

def get_models_small(seed):
    return {
        "cat": CatBoostRegressor(iterations=1500, learning_rate=0.03, depth=4, l2_leaf_reg=8.0,
                rsm=0.3, random_seed=seed, verbose=False, early_stopping_rounds=80),
        "lgb": lgb.LGBMRegressor(n_estimators=1500, learning_rate=0.03, num_leaves=15,
                subsample=0.8, subsample_freq=1, colsample_bytree=0.3, reg_alpha=0.5,
                reg_lambda=3.0, min_child_samples=8, random_state=seed, n_jobs=-1, verbosity=-1),
    }
SMALL_TREE_NAMES = ["cat","lgb"]
SMALL_TRANSFORM  = {"cat":"yj","lgb":"yj"}

# ===================== v8 FIX: honest early stopping =====================
# v7 passed the SCORING fold as eval_set, so the tree count was tuned on the very rows the OOF
# is computed from. Measured inflation on this data, single LightGBM, 8-fold:
#     eea +0.0118 | ei +0.0141 | eps +0.0458 | nc +0.0148 | egb +0.0148 | egc +0.0082
#     (mean +0.0182 -- essentially the whole 0.909 -> 0.887 OOF/LB gap)
# v8 picks the budget on an INNER split of the training fold, then refits on the full training
# fold with that budget and no eval_set at all. The scoring fold is never touched.
INNER_ES_FOLDS = 5

def _budget_on_inner(name, mdl, Xa, ya, seed):
    """Return an iteration budget chosen without ever seeing the scoring fold."""
    ia, ib = next(iter(KFold(INNER_ES_FOLDS, shuffle=True, random_state=seed).split(Xa)))
    if len(ib) < 8:                                  # too few rows to early-stop on
        return None
    try:
        if name.startswith("xgb"):
            mdl.fit(Xa[ia], ya[ia], eval_set=[(Xa[ib], ya[ib])], verbose=False)
            best = getattr(mdl, "best_iteration", None)
        elif name.startswith("lgb"):
            mdl.fit(Xa[ia], ya[ia], eval_set=[(Xa[ib], ya[ib])],
                    callbacks=[lgb.early_stopping(120, verbose=False)])
            best = mdl.best_iteration_
        else:
            mdl.fit(Xa[ia], ya[ia], eval_set=(Xa[ib], ya[ib]))
            best = mdl.get_best_iteration()
    except Exception:
        return None
    if not best or best <= 0:
        return None
    # the refit sees 1/(1-1/k) more rows, so scale the budget up to match
    return max(50, int(best / (1.0 - 1.0/INNER_ES_FOLDS)))

def _refit_full(name, mkfn, seed, n_iter, Xa, ya):
    """Fresh model with the chosen budget, fitted on the whole training fold, no eval_set."""
    m = mkfn(seed)[name]
    if n_iter is not None:
        if name.startswith("cat"): m.set_params(iterations=n_iter, early_stopping_rounds=None)
        elif name.startswith("xgb"): m.set_params(n_estimators=n_iter, early_stopping_rounds=None)
        else: m.set_params(n_estimators=n_iter)
    else:
        if name.startswith("cat"): m.set_params(early_stopping_rounds=None)
        elif name.startswith("xgb"): m.set_params(early_stopping_rounds=None)
    m.fit(Xa, ya)
    return m

# v8: small targets get 10 seeds instead of 3. They are cheap (~220 rows) and their per-target
# LB sampling sd is ~0.025, so averaging more fold partitions is free variance reduction.
BIG_TREE_SEEDS   = [SEED, SEED+1, SEED+2]
SMALL_TREE_SEEDS = [SEED+i for i in range(10)]

def train_trees(target):
    mask = (train["target_type"].values == target)
    XtrF, XteF = get_X(target)                       # polymer blocks only for POLY_TARGETS
    X_full, y = XtrF[mask], train.loc[mask,"target"].values.astype(np.float32)
    groups = train.loc[mask,"smiles_canon"].values
    is_small = target in SMALL_TARGETS
    names   = SMALL_TREE_NAMES if is_small else BIG_TREE_NAMES
    tmap    = SMALL_TRANSFORM  if is_small else BIG_TRANSFORM
    mkfn    = get_models_small if is_small else get_models
    seeds   = SMALL_TREE_SEEDS if is_small else BIG_TREE_SEEDS
    oof = {n: np.zeros(len(y)) for n in names}
    tst = {n: np.zeros(len(test)) for n in names}
    for seed in seeds:
        for tr_i, va_i in make_group_folds(groups, N_FOLDS, seed=seed):
            # feature selection fitted on the TRAINING fold only
            fidx = select_features_infold(X_full[tr_i], y[tr_i], target, seed=seed)
            Xa, Xv, Xt = X_full[tr_i][:,fidx], X_full[va_i][:,fidx], XteF[:,fidx]
            for n in names:
                pt  = _make_t(tmap[n])                       # fitted on the train fold only
                ya  = pt.fit_transform(y[tr_i].reshape(-1,1)).ravel()
                probe  = mkfn(seed)[n]
                budget = _budget_on_inner(n, probe, Xa, ya, seed)
                mdl    = _refit_full(n, mkfn, seed, budget, Xa, ya)
                oof[n][va_i] += pt.inverse_transform(mdl.predict(Xv).reshape(-1,1)).ravel() / len(seeds)
                tst[n]       += pt.inverse_transform(mdl.predict(Xt).reshape(-1,1)).ravel() / (len(seeds)*N_FOLDS)
    blend = np.mean([oof[n] for n in names], axis=0)
    print(f"[{target}] tree blend OOF R2 = {r2_score(y,blend):.4f} | MAE = {mean_absolute_error(y,blend):.3f}"
          f"  ({len(seeds)} seeds)")
    for n in names: print(f"    {n:>7}: {r2_score(y,oof[n]):.4f}")
    return {"oof":oof, "test":tst, "y":y, "mask":mask}

tree_res = {t: train_trees(t) for t in TARGETS}

## 4b. Ridge on dense descriptors only (NEW — extra diversity for the stack)



A plain regularized linear model fit on the *dense* (non-fingerprint) descriptor block. It

can't compete with the trees on raw R2, but because it's linear and ignores fingerprint

bits, its errors are largely decorrelated from tree and GNN errors — which is exactly what

gives the NNLS/Ridge stacker leverage.

In [9]:
from sklearn.preprocessing import StandardScaler



def train_ridge(target):

    mask = (train["target_type"].values == target)

    dense_idx = [i for i,c in enumerate(ALL_COLS) if c in DENSE_COLS]

    X, y = Xtr_all[mask][:, dense_idx], train.loc[mask,"target"].values.astype(np.float32)

    Xt = Xte_all[:, dense_idx]

    groups = train.loc[mask,"smiles_canon"].values

    oof = np.zeros(len(y)); tst = np.zeros(len(test))

    for tr,va in make_group_folds(groups, N_FOLDS, seed=SEED):

        sc = StandardScaler().fit(X[tr])

        model = Ridge(alpha=5.0).fit(sc.transform(X[tr]), y[tr])

        oof[va] = model.predict(sc.transform(X[va]))

        tst += model.predict(sc.transform(Xt)) / N_FOLDS

    print(f"[{target}] ridge OOF R2 = {r2_score(y,oof):.4f}")

    return {"oof":oof, "test":tst}



ridge_res = {t: train_ridge(t) for t in TARGETS}


[eea] ridge OOF R2 = 0.8960
[egb] ridge OOF R2 = 0.8954
[egc] ridge OOF R2 = 0.8613
[ei] ridge OOF R2 = 0.8016
[eps] ridge OOF R2 = 0.8178
[nc] ridge OOF R2 = 0.8729
[tg] ridge OOF R2 = 0.8322


## 5. From-Scratch Message-Passing GNN (pure PyTorch, no torch_geometric)



A graph model reads the molecule directly (atoms = nodes, bonds = edges) instead of hashed

fingerprints, so its errors are decorrelated from the trees — exactly what makes stacking

pay off. Everything is trained from scratch; no external graph library or pretrained weights.

In [10]:
ATOM_LIST = ["C","N","O","S","F","Si","P","Cl","Br","I","B","H","*"]

ATOM_MAP  = {s:i for i,s in enumerate(ATOM_LIST)}

HYB = [Chem.rdchem.HybridizationType.SP, Chem.rdchem.HybridizationType.SP2,

       Chem.rdchem.HybridizationType.SP3, Chem.rdchem.HybridizationType.SP3D,

       Chem.rdchem.HybridizationType.SP3D2]

HYB_MAP = {h:i for i,h in enumerate(HYB)}

BONDS = [Chem.rdchem.BondType.SINGLE, Chem.rdchem.BondType.DOUBLE,

         Chem.rdchem.BondType.TRIPLE, Chem.rdchem.BondType.AROMATIC]

BT_MAP = {b:i for i,b in enumerate(BONDS)}



def _onehot(i,n):

    v=[0.0]*n

    if i is not None and 0<=i<n: v[i]=1.0

    return v

def atom_feat(a):

    return (_onehot(ATOM_MAP.get(a.GetSymbol()), len(ATOM_LIST)) + _onehot(min(a.GetDegree(),5),6)

            + _onehot(min(a.GetTotalNumHs(),4),5) + _onehot(HYB_MAP.get(a.GetHybridization()),len(HYB))

            + [float(a.GetIsAromatic()), float(a.IsInRing()),

               float(a.GetFormalCharge()), float(a.GetSymbol()=="*")])

def bond_feat(b):

    return _onehot(BT_MAP.get(b.GetBondType()),len(BONDS)) + [float(b.GetIsConjugated()), float(b.IsInRing())]

_m = Chem.MolFromSmiles("CC"); ADIM=len(atom_feat(_m.GetAtomWithIdx(0))); BDIM=len(bond_feat(_m.GetBondWithIdx(0)))



def to_graph(smi):

    m = Chem.MolFromSmiles(str(smi))

    if m is None or m.GetNumAtoms()==0: return None

    x = np.array([atom_feat(a) for a in m.GetAtoms()], dtype=np.float32)

    s,d,e = [],[],[]

    for b in m.GetBonds():

        i,j=b.GetBeginAtomIdx(),b.GetEndAtomIdx(); bf=bond_feat(b)

        s+=[i,j]; d+=[j,i]; e+=[bf,bf]

    if not s: s,d,e=[0],[0],[[0.0]*BDIM]

    return x, np.array([s,d],dtype=np.int64), np.array(e,dtype=np.float32)



print("Building molecular graphs...")

G_tr = [to_graph(s) for s in train["smiles"]]

G_te = [to_graph(s) for s in test["smiles"]]

print(f"train graphs ok: {sum(g is not None for g in G_tr)}/{len(G_tr)} | atom_dim={ADIM} bond_dim={BDIM}")



def collate(graphs):

    xs,eis,eas,batch=[],[],[],[]; off=0

    for gi,g in enumerate(graphs):

        x,ei,ea=g; xs.append(x); eis.append(ei+off); eas.append(ea)

        batch += [gi]*x.shape[0]; off += x.shape[0]

    return (torch.tensor(np.concatenate(xs)),

            torch.tensor(np.concatenate(eis,axis=1)),

            torch.tensor(np.concatenate(eas)),

            torch.tensor(batch,dtype=torch.int64))


Building molecular graphs...
train graphs ok: 7405/7405 | atom_dim=33 bond_dim=6


In [ ]:
class MPNN(nn.Module):
    def __init__(self, adim, bdim, hid=160, layers=4, drop=0.1):
        super().__init__(); self.L=layers
        self.lin0=nn.Linear(adim,hid); self.edge=nn.Linear(bdim,hid)
        self.msg=nn.ModuleList([nn.Linear(2*hid,hid) for _ in range(layers)])
        self.upd=nn.ModuleList([nn.GRUCell(hid,hid) for _ in range(layers)])
        self.bn =nn.ModuleList([nn.BatchNorm1d(hid) for _ in range(layers)])
        self.head=nn.Sequential(nn.Linear(2*hid,hid),nn.ReLU(),nn.Dropout(drop),
                                nn.Linear(hid,hid//2),nn.ReLU(),nn.Linear(hid//2,1))
    def forward(self,X,EI,EA,B):
        h=F.relu(self.lin0(X)); e=self.edge(EA); s,d=EI[0],EI[1]
        for l in range(self.L):
            msg=torch.relu(self.msg[l](torch.cat([h[s],e],1)))
            agg=torch.zeros_like(h).index_add_(0,d,msg)
            h=self.bn[l](self.upd[l](agg,h))
        ng=int(B.max().item())+1
        summ=torch.zeros(ng,h.size(1),device=h.device).index_add_(0,B,h)
        cnt=torch.zeros(ng,1,device=h.device).index_add_(0,B,torch.ones(h.size(0),1,device=h.device))
        mean=summ/cnt.clamp(min=1)
        mx=torch.full((ng,h.size(1)),-1e9,device=h.device).index_reduce_(0,B,h,"amax",include_self=True)
        return self.head(torch.cat([mean,mx],1)).squeeze(-1)

def gnn_predict(model, graphs, bs=256):
    model.eval(); out=np.zeros(len(graphs),dtype=np.float32)
    with torch.no_grad():
        for i in range(0,len(graphs),bs):
            gs=graphs[i:i+bs]; X,EI,EA,B=collate(gs)
            X,EI,EA,B=X.to(DEVICE),EI.to(DEVICE),EA.to(DEVICE),B.to(DEVICE)
            out[i:i+bs]=model(X,EI,EA,B).cpu().numpy()
    return out

# ===================== v8 FIX: honest checkpoint selection =====================
# v7 kept the epoch with the best R2 ON THE SCORING FOLD (`best_r2 = r2_score(yva, vp)`), then
# reported that fold's predictions as OOF. With 8 folds on a ~220-row target the scoring fold is
# ~27 rows, and taking the argmax over up to 180 epochs of a 27-row R2 is a large upward bias.
# Monte-Carlo on this geometry (rho=0.92, AR(1) plateau jitter) puts it at +0.042 to +0.071 for
# the ~220-row targets and +0.035 to +0.060 for egb -- far larger than the tree leak, and the
# reason NNLS was over-weighting the GNN columns on test.
#
# v8 carves an INNER validation split out of the training fold for early stopping, and averages
# the last SWA_LAST epochs' weights instead of picking a single best checkpoint. Averaging also
# removes the epoch-to-epoch jitter that the old rule was exploiting. The scoring fold is used
# for nothing but scoring.
INNER_VAL_FRAC = 0.15
SWA_LAST       = 10

def train_gnn_fold(gtr, ytr, gva, yva, epochs=180, bs=64, lr=5e-4, patience=30, seed=SEED):
    rng = np.random.RandomState(seed)
    perm = rng.permutation(len(gtr))
    n_in = max(8, int(INNER_VAL_FRAC*len(gtr)))
    iva, itr = perm[:n_in], perm[n_in:]
    g_in  = [gtr[i] for i in itr]; y_in  = ytr[itr]
    g_iva = [gtr[i] for i in iva]; y_iva = ytr[iva]

    mu, sd = y_in.mean(), y_in.std()+1e-8
    y_in_n = (y_in-mu)/sd
    model = MPNN(ADIM,BDIM).to(DEVICE)
    if "PRETRAINED_STATE" in globals():
        _md=model.state_dict()
        for _k,_v in PRETRAINED_STATE.items():
            if _k in _md and _md[_k].shape==_v.shape: _md[_k]=_v
        model.load_state_dict(_md)
    opt=torch.optim.Adam(model.parameters(),lr=lr,weight_decay=1e-5)
    sched=torch.optim.lr_scheduler.CosineAnnealingLR(opt,T_max=epochs)
    idx=np.arange(len(g_in)); best=-1e9; wait=0; swa=[]
    for ep in range(epochs):
        model.train(); np.random.shuffle(idx)
        for i in range(0,len(idx),bs):
            bi=idx[i:i+bs]
            if len(bi)<2: continue
            X,EI,EA,B=collate([g_in[j] for j in bi])
            X,EI,EA,B=X.to(DEVICE),EI.to(DEVICE),EA.to(DEVICE),B.to(DEVICE)
            pred=model(X,EI,EA,B)
            loss=F.smooth_l1_loss(pred, torch.tensor(y_in_n[bi],device=DEVICE))
            opt.zero_grad(); loss.backward()
            torch.nn.utils.clip_grad_norm_(model.parameters(),5.0); opt.step()
        sched.step()
        swa.append({k:v.detach().cpu().clone() for k,v in model.state_dict().items()})
        if len(swa)>SWA_LAST: swa.pop(0)
        r2_in = r2_score(y_iva, gnn_predict(model,g_iva)*sd+mu)   # INNER split, not the scoring fold
        if r2_in>best: best=r2_in; wait=0
        else:
            wait+=1
            if wait>=patience: break
    # stochastic weight averaging over the tail -- no single-checkpoint selection anywhere
    avg={k: torch.stack([s[k].float() for s in swa]).mean(0) for k in swa[0]}
    for k,v in model.state_dict().items():
        if v.dtype not in (torch.float32, torch.float64): avg[k]=v.cpu()
    model.load_state_dict(avg, strict=False)     # CPU tensors copy into CUDA params fine
    model.to(DEVICE)
    return model, mu, sd, best

def train_gnn(target, seeds=(SEED,SEED+1)):
    mask = (train["target_type"].values == target)
    idxs = np.where(mask)[0]
    y = train.loc[mask,"target"].values.astype(np.float32)
    groups = train.loc[mask,"smiles_canon"].values
    gtr_all = [G_tr[i] for i in idxs]
    oof = np.zeros(len(y)); tst = np.zeros(len(test)); n_runs=0
    for seed in seeds:
        for tr_i,va_i in make_group_folds(groups, N_FOLDS, seed=seed):
            model,mu,sd,_ = train_gnn_fold([gtr_all[i] for i in tr_i], y[tr_i],
                                           [gtr_all[i] for i in va_i], y[va_i], seed=seed)
            oof[va_i] += (gnn_predict(model,[gtr_all[i] for i in va_i])*sd+mu)/len(seeds)
            tst += (gnn_predict(model,G_te)*sd+mu)/(len(seeds)*N_FOLDS)
            n_runs += 1
    print(f"[{target}] GNN OOF R2 = {r2_score(y,oof):.4f}  ({n_runs} fold-runs, honest checkpointing)")
    return {"oof":oof, "test":tst}

GNN_TARGETS = list(BIG_TARGETS) + [t for t in ["ei","eps"] if t in TARGETS and t not in BIG_TARGETS]
print("Per-target GNN targets:", GNN_TARGETS)

## 5c. Self-supervised PI1M pretraining + multi-task shared-encoder GNN (added)



Two additions that target the phase-2 structure directly:



* **PI1M pretraining.** PI1M.csv (~995k unlabeled polymer SMILES) is the *auxiliary data* in the

  competition's data section. The rules ban publicly-available external data and pretrained models but

  explicitly permit the provided auxiliary data, so training our OWN encoder from scratch on PI1M

  (masked-atom-type + self-computed-descriptor regression, no labels, no imported weights) is compliant.

* **Multi-task GNN.** One shared message-passing encoder, seven heads (one per property), masked loss so

  each molecule trains only the heads it has labels for. The 4,143 `tg` and 2,028 `egc` rows shape a

  general embedding the ~220-row targets ride on — the transfer the per-target models can't get. Its OOF

  column enters every target's stack as `mtl`, and is typically the strongest single member on the small

  targets. Encoder warm-started from the PI1M pretraining.

In [ ]:
# ===== self-supervised PI1M pretraining, then multi-task GNN with 7 masked heads =====
USE_PI1M = True            # RULES-CONFIRMED: PI1M is provided auxiliary data (not external / not pretrained weights)
PI1M_MAX = 60000           # subsample for tractable pretraining; lower if tight on the time limit
PRETRAIN_EPOCHS = 40
NATOM = len(ATOM_LIST); T_ALL = len(TARGETS); _tidx = {t:i for i,t in enumerate(TARGETS)}

def _aux_vec(m):
    if m is None: return None
    na = m.GetNumAtoms() or 1
    arom = sum(a.GetIsAromatic() for a in m.GetAtoms())/na
    return [Descriptors.MolWt(m), Descriptors.TPSA(m), Descriptors.MolLogP(m),
            rdMolDescriptors.CalcFractionCSP3(m), rdMolDescriptors.CalcNumRotatableBonds(m),
            rdMolDescriptors.CalcNumHBD(m), rdMolDescriptors.CalcNumHBA(m), arom,
            rdMolDescriptors.CalcNumAromaticRings(m), rdMolDescriptors.CalcNumRings(m),
            rdMolDescriptors.CalcNumHeteroatoms(m), Descriptors.LabuteASA(m)]
NAUX = 12

# build the pretraining corpus: train graphs (+ PI1M subset if enabled)
_pre_graphs, _pre_aux = [], []
for g, s in zip(G_tr, train["smiles"].values):
    m = Chem.MolFromSmiles(str(s)); a = _aux_vec(m)
    if g is not None and a is not None: _pre_graphs.append(g); _pre_aux.append(a)
if USE_PI1M and os.path.exists(PI1M_PATH):
    _pdf = pd.read_csv(PI1M_PATH); _pcol = "SMILES" if "SMILES" in _pdf.columns else _pdf.columns[0]
    _psm = _pdf[_pcol].astype(str).values
    if len(_psm) > PI1M_MAX:
        _rng = np.random.RandomState(SEED); _psm = _psm[_rng.choice(len(_psm), PI1M_MAX, replace=False)]
    print(f"Adding {len(_psm)} PI1M SMILES to the pretraining corpus ...")
    for s in _psm:
        m = Chem.MolFromSmiles(str(s))
        if m is None: continue
        g = to_graph(s); a = _aux_vec(m)
        if g is not None and a is not None: _pre_graphs.append(g); _pre_aux.append(a)
elif USE_PI1M:
    print("USE_PI1M=True but PI1M.csv not found -> pretraining on train SMILES only.")
_pre_aux = np.array(_pre_aux, np.float32)
_aux_mu, _aux_sd = _pre_aux.mean(0), _pre_aux.std(0)+1e-6
_pre_aux = (_pre_aux - _aux_mu)/_aux_sd
print(f"Pretraining corpus: {len(_pre_graphs)} molecules")

class MultiTaskMPNN(nn.Module):
    """Shared MPNN encoder; encode() names match the pretraining encoder for 1:1 warm-start."""
    def __init__(self, adim, bdim, hid=192, layers=4, drop=0.1, n_tasks=T_ALL):
        super().__init__(); self.L=layers
        self.lin0=nn.Linear(adim,hid); self.edge=nn.Linear(bdim,hid)
        self.msg=nn.ModuleList([nn.Linear(2*hid,hid) for _ in range(layers)])
        self.upd=nn.ModuleList([nn.GRUCell(hid,hid) for _ in range(layers)])
        self.bn =nn.ModuleList([nn.BatchNorm1d(hid) for _ in range(layers)])
        self.trunk=nn.Sequential(nn.Linear(2*hid,hid),nn.ReLU(),nn.Dropout(drop))
        self.heads=nn.ModuleList([nn.Sequential(nn.Linear(hid,hid//2),nn.ReLU(),nn.Linear(hid//2,1))
                                  for _ in range(n_tasks)])
        self.node_head=nn.Linear(hid,NATOM); self.graph_head=nn.Linear(hid,NAUX)   # pretrain heads
    def _prop(self,X,EI,EA,B):
        h=F.relu(self.lin0(X)); e=self.edge(EA); s,d=EI[0],EI[1]
        for l in range(self.L):
            msg=torch.relu(self.msg[l](torch.cat([h[s],e],1)))
            agg=torch.zeros_like(h).index_add_(0,d,msg)
            h=self.bn[l](self.upd[l](agg,h))
        ng=int(B.max().item())+1
        summ=torch.zeros(ng,h.size(1),device=h.device).index_add_(0,B,h)
        cnt=torch.zeros(ng,1,device=h.device).index_add_(0,B,torch.ones(h.size(0),1,device=h.device))
        mean=summ/cnt.clamp(min=1)
        mx=torch.full((ng,h.size(1)),-1e9,device=h.device).index_reduce_(0,B,h,"amax",include_self=True)
        return h, self.trunk(torch.cat([mean,mx],1)), mean
    def forward(self,X,EI,EA,B):
        _,z,_=self._prop(X,EI,EA,B)
        return torch.cat([hd(z) for hd in self.heads],dim=1)
    def pretrain_forward(self,X,EI,EA,B):
        h,_,mean=self._prop(X,EI,EA,B)
        return self.node_head(h), self.graph_head(mean)

def pretrain_encoder(epochs=PRETRAIN_EPOCHS, bs=256, lr=5e-4, mask_rate=0.15, aux_w=0.5, seed=SEED):
    torch.manual_seed(seed); np.random.seed(seed)
    net=MultiTaskMPNN(ADIM,BDIM).to(DEVICE)
    opt=torch.optim.Adam(net.parameters(),lr=lr,weight_decay=1e-5)
    sched=torch.optim.lr_scheduler.CosineAnnealingLR(opt,T_max=epochs)
    ce=nn.CrossEntropyLoss(); hub=nn.SmoothL1Loss()
    idx=np.arange(len(_pre_graphs))
    for ep in range(epochs):
        net.train(); np.random.shuffle(idx); tot=0.0
        for i2 in range(0,len(idx),bs):
            bi=idx[i2:i2+bs]
            if len(bi)<2: continue
            X,EI,EA,B=collate([_pre_graphs[j] for j in bi])
            X,EI,EA,B=X.to(DEVICE),EI.to(DEVICE),EA.to(DEVICE),B.to(DEVICE)
            aux=torch.tensor(_pre_aux[bi],device=DEVICE)
            n=X.size(0); nmask=max(1,int(mask_rate*n)); midx=torch.randperm(n,device=DEVICE)[:nmask]
            labels=X[midx,:NATOM].argmax(1)
            Xc=X.clone(); Xc[midx,:NATOM]=0.0
            nl,gp=net.pretrain_forward(Xc,EI,EA,B)
            loss=ce(nl[midx],labels)+aux_w*hub(gp,aux)
            opt.zero_grad(); loss.backward()
            torch.nn.utils.clip_grad_norm_(net.parameters(),5.0); opt.step(); tot+=float(loss)
        sched.step()
        if (ep+1)%10==0 or ep==0: print(f"  pretrain {ep+1}/{epochs} loss={tot/max(1,len(idx)//bs):.4f}")
    return {k:v.detach().cpu().clone() for k,v in net.state_dict().items()
            if k.split(".")[0] in ("lin0","edge","msg","upd","bn")}

print("Pretraining shared encoder ...")
PRETRAINED_STATE = pretrain_encoder()
def apply_pretrained(model):
    md_=model.state_dict(); n=0
    for k,v in PRETRAINED_STATE.items():
        if k in md_ and md_[k].shape==v.shape: md_[k]=v; n+=1
    model.load_state_dict(md_); return n

_MT_MU=np.array([train.loc[train.target_type==t,"target"].mean() for t in TARGETS],np.float32)
_MT_SD=np.array([train.loc[train.target_type==t,"target"].std()+1e-6 for t in TARGETS],np.float32)

def _mt_predict(model,graphs,bs=256):
    model.eval(); out=np.zeros((len(graphs),T_ALL),np.float32)
    with torch.no_grad():
        for i2 in range(0,len(graphs),bs):
            X,EI,EA,B=collate(graphs[i2:i2+bs])
            X,EI,EA,B=X.to(DEVICE),EI.to(DEVICE),EA.to(DEVICE),B.to(DEVICE)
            out[i2:i2+bs]=model(X,EI,EA,B).cpu().numpy()
    return out

def train_multitask(seeds=(SEED,SEED+1),epochs=200,bs=64,lr=5e-4,patience=30):
    groups_all=train["smiles_canon"].values; tt=train["target_type"].values
    y=train["target"].values.astype(np.float32)
    ymat=np.full((len(train),T_ALL),np.nan,np.float32)
    for i2 in range(len(train)): ymat[i2,_tidx[tt[i2]]]=y[i2]
    ynorm=(ymat-_MT_MU)/_MT_SD
    row_of={t:np.where(tt==t)[0] for t in TARGETS}
    pos_in={t:{r:i for i,r in enumerate(row_of[t])} for t in TARGETS}
    oof={t:np.full(len(row_of[t]),np.nan,np.float32) for t in TARGETS}
    tst={t:np.zeros(len(test),np.float32) for t in TARGETS}
    for seed in seeds:
        torch.manual_seed(seed); np.random.seed(seed)
        for fi_,(tr,va) in enumerate(make_group_folds(groups_all,N_FOLDS,seed=seed)):
            model=MultiTaskMPNN(ADIM,BDIM).to(DEVICE); apply_pretrained(model)
            opt=torch.optim.AdamW(model.parameters(),lr=lr,weight_decay=1e-5)
            sched=torch.optim.lr_scheduler.CosineAnnealingLR(opt,T_max=epochs)
            # carve an inner selection split out of the TRAINING fold
            _rng=np.random.RandomState(seed); _perm=_rng.permutation(len(tr))
            _nin=max(20,int(0.15*len(tr))); isel=_perm[:_nin]; ifit=_perm[_nin:]
            g_isel=[G_tr[tr[i]] for i in isel]
            gtr=[G_tr[i] for i in tr]; Ytr=ynorm[tr]
            Mtr=torch.tensor(~np.isnan(Ytr)); Ytr0=torch.tensor(np.nan_to_num(Ytr))
            gva=[G_tr[i] for i in va]; idx=np.array(ifit); best=-1e9; best_state=None; wait=0
            for ep in range(epochs):
                model.train(); np.random.shuffle(idx)
                for k in range(0,len(idx),bs):
                    bi=idx[k:k+bs]
                    if len(bi)<2: continue
                    X,EI,EA,B=collate([gtr[j] for j in bi])
                    X,EI,EA,B=X.to(DEVICE),EI.to(DEVICE),EA.to(DEVICE),B.to(DEVICE)
                    pred=model(X,EI,EA,B); mmask=Mtr[bi].to(DEVICE); tgt=Ytr0[bi].to(DEVICE)
                    diff=(pred-tgt)[mmask]
                    loss=F.smooth_l1_loss(diff,torch.zeros_like(diff))
                    opt.zero_grad(); loss.backward()
                    torch.nn.utils.clip_grad_norm_(model.parameters(),5.0); opt.step()
                sched.step()
                # v8: model selection on an INNER split of the training fold, never on `va`.
                # v7 scored `va` here and kept its argmax epoch, then reported `va` as OOF --
                # worth +0.04..+0.07 of pure optimism at 27-row folds.
                vp=_mt_predict(model,g_isel)*_MT_SD+_MT_MU; r2s=[]
                for j,t in enumerate(TARGETS):
                    rows=[k2 for k2,r in enumerate(isel) if tt[tr[r]]==t]
                    if len(rows)>=5:
                        yr=y[[tr[isel[k2]] for k2 in rows]]; pr=vp[rows,j]
                        r2s.append(r2_score(yr,pr))
                r2=float(np.mean(r2s)) if r2s else -1e9
                if r2>best: best=r2; best_state={k:v.cpu().clone() for k,v in model.state_dict().items()}; wait=0
                else:
                    wait+=1
                    if wait>=patience: break
            model.load_state_dict(best_state)
            vp=_mt_predict(model,gva)*_MT_SD+_MT_MU
            for li,r in enumerate(va):
                t=tt[r]; j=_tidx[t]; p=pos_in[t][r]
                oof[t][p]=(0.0 if np.isnan(oof[t][p]) else oof[t][p])+vp[li,j]/len(seeds)
            tp=_mt_predict(model,G_te)*_MT_SD+_MT_MU
            for j,t in enumerate(TARGETS): tst[t]+=tp[:,j]/(len(seeds)*N_FOLDS)
    res={}
    for t in TARGETS:
        yt=y[row_of[t]]; v=~np.isnan(oof[t])
        print(f"[{t}] multitask OOF R2 = {r2_score(yt[v],oof[t][v]):.4f}  ({int(v.sum())} rows)")
        res[t]={"oof":oof[t],"test":tst[t],"y":yt}
    return res

print("Training multi-task GNN ...")
mtl_res = train_multitask()

# ===== v7: dedicated per-target GNNs (now warm-started from PRETRAINED_STATE) =====
# Built here, after pretraining, so BIG targets AND the weak ei/eps get a transfer-seeded GNN member.
print("Training dedicated per-target GNNs (warm-started) ...")
gnn_res = {t: train_gnn(t) for t in GNN_TARGETS}

## 5d. Tanimoto/MinMax-kernel GP — the measured +0.010 base learner

In [ ]:
# ===== 5d. Tanimoto / MinMax-kernel Gaussian Process (NEW in v8) =====
# The single most reliable addition measured on this data. A GP with a fingerprint similarity
# kernel is decorrelated from both the trees (no axis-aligned splits) and the GNN (no learned
# representation), and 200-row chemistry regression is exactly where kernel methods are strong.
#
# MEASURED blend gain over a tree-only baseline, honest 8-fold:
#     eea +0.0255 | egb +0.0232 | ei +0.0103 | eps +0.0043 | nc +0.0008
# Note ei: the GP alone scores 0.8022 vs the tree's 0.8109, yet the blend reaches 0.8212. The
# value is decorrelation, not standalone accuracy -- so keep it in the stack even where its own
# OOF looks unimpressive.
#
# Kernel: sum of MinMax(Morgan counts) -- the count-vector generalisation of Tanimoto, PSD -- and
# an RBF on the standardised dense descriptors. The dense half is what lets the GP see the
# cross-target sibling block, which is the strongest signal on the small targets and invisible to
# a purely structural kernel. Hyper-parameters by exact log marginal likelihood on the training
# fold; the noise floor is bounded away from 0 so the GP cannot interpolate.
from scipy.linalg import cho_factor, cho_solve
from sklearn.preprocessing import StandardScaler

_M2_IDX = np.array([i for i,c in enumerate(ALL_COLS) if c.startswith("m2_")])
_DENSE_IDX = np.array([i for i,c in enumerate(ALL_COLS) if c in set(DENSE_COLS)])

def _minmax_K(A,B, block=256):
    K=np.empty((A.shape[0],B.shape[0]),dtype=np.float64)
    for i in range(0,A.shape[0],block):
        chunk=A[i:i+block]
        for r in range(chunk.shape[0]):
            K[i+r]=np.minimum(chunk[r],B).sum(1)/np.maximum(np.maximum(chunk[r],B).sum(1),1e-9)
    return K

def _tanimoto_K(A,B):
    """Fast binary Tanimoto via one matmul -- used on the big targets where MinMax is O(n^2 d)."""
    Ab=(A>0).astype(np.float64); Bb=(B>0).astype(np.float64)
    inter=Ab@Bb.T
    na=Ab.sum(1)[:,None]; nb=Bb.sum(1)[None,:]
    return inter/np.maximum(na+nb-inter,1e-9)

def _gp_fit_predict(Ka, ya, Kb, amp_grid=(0.5,1.0,2.0), noise_grid=(3e-3,1e-2,3e-2,1e-1,3e-1)):
    mu,sd=ya.mean(),ya.std()+1e-9; yn=(ya-mu)/sd; n=len(ya)
    best=(-1e18,None)
    for amp in amp_grid:
        for nz in noise_grid:
            try: c=cho_factor(amp*Ka+nz*np.eye(n), lower=True)
            except Exception: continue
            al=cho_solve(c,yn)
            lml=-0.5*float(yn@al)-float(np.log(np.diag(c[0])).sum())
            if lml>best[0]: best=(lml,(amp,al))
    if best[1] is None: return np.full(Kb.shape[0], mu)
    amp,al=best[1]
    return (amp*Kb)@al*sd+mu

def train_gp(target):
    mask=(train["target_type"].values==target)
    XtrF,XteF=get_X(target)
    y=train.loc[mask,"target"].values.astype(np.float64)
    groups=train.loc[mask,"smiles_canon"].values
    is_small=target in SMALL_TARGETS
    FPtr=XtrF[mask][:,_M2_IDX]; FPte=XteF[:,_M2_IDX]
    didx=_DENSE_IDX
    if target in POLY_TARGETS:                       # poly cols sit past len(ALL_COLS)
        didx=np.concatenate([_DENSE_IDX, np.arange(len(ALL_COLS), XtrF.shape[1])])
    Dtr =XtrF[mask][:,didx]; Dte=XteF[:,didx]
    kfun=_minmax_K if is_small else _tanimoto_K
    grids=dict() if is_small else dict(amp_grid=(1.0,), noise_grid=(1e-2,3e-2,1e-1))
    oof=np.zeros(len(y)); tst=np.zeros(len(test))
    for tr_i,va_i in make_group_folds(groups, N_FOLDS, seed=SEED):
        sc=StandardScaler().fit(Dtr[tr_i])
        Za=np.clip(sc.transform(Dtr[tr_i]),-8,8)
        Zv=np.clip(sc.transform(Dtr[va_i]),-8,8)
        Zt=np.clip(sc.transform(Dte),-8,8)
        g=1.0/max(Za.shape[1],1)
        sa=(Za**2).sum(1)
        Ra=np.exp(-g*(sa[:,None]+sa[None,:]-2*Za@Za.T))
        Rv=np.exp(-g*((Zv**2).sum(1)[:,None]+sa[None,:]-2*Zv@Za.T))
        Rt=np.exp(-g*((Zt**2).sum(1)[:,None]+sa[None,:]-2*Zt@Za.T))
        Ka=0.5*kfun(FPtr[tr_i],FPtr[tr_i])+0.5*Ra
        Kv=0.5*kfun(FPtr[va_i],FPtr[tr_i])+0.5*Rv
        Kt=0.5*kfun(FPte,       FPtr[tr_i])+0.5*Rt
        oof[va_i]=_gp_fit_predict(Ka,y[tr_i],Kv,**grids)
        tst      +=_gp_fit_predict(Ka,y[tr_i],Kt,**grids)/N_FOLDS
    print(f"[{target}] GP OOF R2 = {r2_score(y,oof):.4f}  ({'MinMax counts' if is_small else 'binary Tanimoto'})")
    return {"oof":oof,"test":tst}

print("Training Tanimoto/MinMax-kernel GPs ...")
gp_res = {t: train_gp(t) for t in TARGETS}

## 6. Stacking: shrunk non-negative blend of trees / ridge / GNN / multitask / GP

Weights come from NNLS on the OOF matrix, then get **shrunk toward equal weighting** by a factor
chosen with nested CV. On this data the inner CV picked pure equal weighting (lam=1.0) for eea, ei
and egb -- with ~220 rows and 6-8 members, fitted weights are mostly noise. The v7 `single`
fallback is removed: it picked its member by scoring the full OOF and then re-scored that same
choice in the meta-CV.

In [ ]:
# ===================== v8 stacking =====================
# Two changes, both measured:
#  1. Shrinkage toward equal weights. With ~220 rows and 6-8 base columns, NNLS weights are
#     mostly noise. Sweeping lambda in the shrunk combiner w = (1-lam)*w_nnls + lam/K, the inner
#     CV picked lam=1.0 (i.e. PURE equal weighting) on eea, ei and egb, and lam=0.5 on eps.
#     Learned weights only earned their keep on nc (lam=0.0).
#  2. The "single" option is gone. best_single was chosen by scoring every member on the FULL
#     OOF and then evaluated in the same meta-CV as if it were a fixed choice -- leaky, and very
#     high variance at this sample size.

def _shrunk_weights(Mtr, ytr, lam):
    K = Mtr.shape[1]
    w,_ = nnls(Mtr, ytr)
    w = w/w.sum() if w.sum() > 0 else np.ones(K)/K
    return (1.0-lam)*w + lam*np.ones(K)/K

LAM_GRID = [0.0, 0.25, 0.5, 0.75, 1.0]

def stack(target):
    y = tree_res[target]["y"]
    _tn = list(tree_res[target]["oof"].keys())
    base      = {n: tree_res[target]["oof"][n]  for n in _tn}
    test_base = {n: tree_res[target]["test"][n] for n in _tn}
    base["ridge"] = ridge_res[target]["oof"]; test_base["ridge"] = ridge_res[target]["test"]
    base["mtl"]   = mtl_res[target]["oof"];   test_base["mtl"]   = mtl_res[target]["test"]
    if target in gnn_res:
        base["gnn"] = gnn_res[target]["oof"]; test_base["gnn"] = gnn_res[target]["test"]
    if target in gp_res:
        base["gp"]  = gp_res[target]["oof"];  test_base["gp"]  = gp_res[target]["test"]
    names=list(base.keys())
    valid=np.ones(len(y),bool)
    for n in names: valid &= ~np.isnan(base[n])
    M=np.column_stack([base[n][valid] for n in names]); yv=y[valid]
    for n in names: print(f"[{target}] {n:>5} OOF R2 = {r2_score(yv,base[n][valid]):.4f}")

    kf=KFold(n_splits=5, shuffle=True, random_state=SEED)
    best=(-1e9,None)
    for lam in LAM_GRID:
        cv=np.zeros(len(yv))
        for tr_i,ev_i in kf.split(M):
            cv[ev_i]=M[ev_i]@_shrunk_weights(M[tr_i], yv[tr_i], lam)
        s=r2_score(yv,cv)
        if s>best[0]: best=(s,lam)
    r2_hat,lam=best
    w=_shrunk_weights(M,yv,lam)
    print(f"[{target}] meta-CV R2={r2_hat:.4f}  lam={lam}  weights="
          +" ".join(f"{n}={wi:.2f}" for n,wi in zip(names,w)))
    Tm=np.column_stack([test_base[n] for n in names])
    return r2_hat, Tm@w

final={}; oof_scores={}
for t in TARGETS:
    r2,tp=stack(t); final[t]=tp; oof_scores[t]=r2
mean_oof=np.mean(list(oof_scores.values()))
print("\nHONEST FINAL OOF (nested meta-CV, honest base columns) per target:")
for t in TARGETS: print(f"   {t:>4}: {oof_scores[t]:.4f}")
print(f"   mean R2 (competition metric) = {mean_oof:.4f}")
print("\nNOTE: this number is LOWER than v7's 0.909 and that is the point -- v7's base columns")
print("were fitted on the rows they were scored on. Expect this to track the leaderboard within")
print("~0.006 (the metric's own sampling sd on these test sizes) instead of sitting 0.022 above it.")

## 7. Build Submission

In [14]:
pred=np.full(len(test), np.nan)

for t in TARGETS:

    m=(test["target_type"].values==t)

    pred[m]=final[t][m]

for t in TARGETS:

    m=(test["target_type"].values==t)&np.isnan(pred)

    if m.any(): pred[m]=train.loc[train["target_type"]==t,"target"].median()



sub=pd.DataFrame({"id":test["id"].values, "target":pred})

sub.to_csv("submission.csv", index=False)

print("Saved submission.csv", sub.shape)

sub.head()


Saved submission.csv (4940, 2)


,id,target
0,1,4.168240
1,2,2.443394
2,3,311.150339
3,4,-37.226479
4,5,4.200862
